# Visual–Temporal Time Cell Modeling (Variable-Length Trajectories)
读取 **可变长度轨迹数据**（每条轨迹独立文件夹，包含 pkl + 图像序列），并结合 CNN + LSTM 进行视觉时间联合建模。

In [ ]:

import os
import pickle
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class VariableLengthTrajectoryDataset(Dataset):
    def __init__(self, root_dir, transform=None, use_yaw=True, min_len=1):
        self.root_dir = root_dir
        self.use_yaw = use_yaw
        self.min_len = min_len
        self.trajectories = sorted([
            os.path.join(root_dir, d)
            for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.transform = transform or transforms.Compose([
            transforms.Resize((64,64)),
            transforms.ToTensor()
        ])
        print(f"Found {len(self.trajectories)} trajectories in {root_dir}")

        # 自动识别输出维度
        sample_traj = os.path.join(self.trajectories[0], "traj_data.pkl")
        with open(sample_traj, 'rb') as f:
            sample_data = pickle.load(f)
        base_dim = len(sample_data['position'][0])
        self.target_dim = base_dim + (1 if self.use_yaw and 'yaw' in sample_data else 0)
        print(f"Detected target_dim = {self.target_dim} (position + yaw={self.use_yaw})")

    def __len__(self):
        return len(self.trajectories)

    def _load_traj(self, traj_path):
        pkl_path = os.path.join(traj_path, "traj_data.pkl")
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        pos = np.array(data['position'], dtype=np.float32)
        yaw = np.array(data['yaw'], dtype=np.float32)
        return pos, yaw

    def _load_imgs(self, traj_path):
        # img_dir = os.path.join(traj_path)
        img_dir = traj_path
        img_files = sorted([
            os.path.join(traj_path, f)
            for f in os.listdir(traj_path)
            if f.lower().endswith(('.png', '.jpg'))
        ])
        imgs = [self.transform(Image.open(f).convert('RGB')) for f in img_files]
        return imgs

    def __getitem__(self, idx):
        traj_path = self.trajectories[idx]
        pos, yaw = self._load_traj(traj_path)
        imgs = self._load_imgs(traj_path)
        n = min(len(pos), len(imgs))
        pos, yaw, imgs = pos[:n], yaw[:n], imgs[:n]
        if n < self.min_len:
            raise ValueError(f"Trajectory {traj_path} too short ({n} < {self.min_len})")
        feats = np.concatenate([pos, yaw[:, None]], axis=1) if self.use_yaw else pos
        imgs_tensor = torch.stack(imgs)
        target = torch.tensor(pos[-1], dtype=torch.float32)
        feats_tensor = torch.tensor(feats, dtype=torch.float32)
        return feats_tensor, imgs_tensor, target


In [ ]:

def collate_variable_length(batch):
    feats, imgs, targets = zip(*batch)
    lengths = [f.shape[0] for f in feats]
    max_len = max(lengths)
    feat_dim = feats[0].shape[1]
    feats_padded = torch.zeros(len(batch), max_len, feat_dim)
    img_dim = imgs[0].shape[1:]
    imgs_padded = torch.zeros(len(batch), max_len, *img_dim)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, (f, im) in enumerate(zip(feats, imgs)):
        L = f.shape[0]
        feats_padded[i, :L] = f
        imgs_padded[i, :L] = im
        mask[i, :L] = True
    targets = torch.stack(targets)
    return feats_padded, imgs_padded, targets, mask


In [ ]:

import torch.nn as nn
import torch.nn.functional as F

class VisualTemporalNet(nn.Module):
    def __init__(self, feature_dim=4, hidden_dim=128, visual_dim=256, output_dim=None):
        super().__init__()
        self.visual_dim = visual_dim
        self.hidden_dim = hidden_dim
        self.feature_dim = feature_dim
        self.output_dim = output_dim if output_dim is not None else feature_dim

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, visual_dim), nn.ReLU()
        )

        # 注意：此处不立即创建 temporal_conv，在 forward 时动态构造
        self.temporal_conv = None  
        self.attn = nn.MultiheadAttention(embed_dim=visual_dim, num_heads=4, batch_first=True)
        self.lstm = nn.LSTM(visual_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, feature_dim)

    def forward(self, feats, imgs, mask):
        # Ensure model components are on the same device as input tensors
        # This is generally handled by model.to(device) initially, but good practice for dynamic layers
        # current_device = imgs.device
        # self.cnn.to(current_device)
        # self.attn.to(current_device)
        # self.lstm.to(current_device)
        # self.fc.to(current_device)

        B, T, _, _, _ = imgs.shape
        visual_feats = []
        for t in range(T):
            vis = self.cnn(imgs[:, t])
            visual_feats.append(vis)
        visual_feats = torch.stack(visual_feats, dim=1)

        # feats = feats.to(current_device)

        # 如果某条样本少一个通道，这里修正 feature_dim
        feat_dim_actual = feats.shape[-1]
        combined = torch.cat([visual_feats, feats], dim=-1)

        # 动态构造 temporal_conv
        if self.temporal_conv is None or self.temporal_conv.in_channels != self.visual_dim + feat_dim_actual:
            self.temporal_conv = nn.Conv1d(self.visual_dim + feat_dim_actual, self.visual_dim, kernel_size=3, padding=1)
            if next(self.parameters()).is_cuda:
                self.temporal_conv = self.temporal_conv.cuda()

        combined = combined.transpose(1, 2)
        combined = F.relu(self.temporal_conv(combined))
        combined = combined.transpose(1, 2)
        attn_out, _ = self.attn(combined, combined, combined, key_padding_mask=~mask)
        packed = nn.utils.rnn.pack_padded_sequence(attn_out, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
        pred = self.fc(lstm_out[range(B), mask.sum(1)-1])
        return pred, lstm_out




In [ ]:

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

dataset = VariableLengthTrajectoryDataset('/media/zhen/Data/Datasets/nomad_data/go_stanford', min_len=1)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_variable_length)

model = VisualTemporalNet(feature_dim=4, hidden_dim=128, visual_dim=256, output_dim=dataset.target_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
num_epochs = 50

def safe_loss(pred, target, loss_fn):
    min_dim = min(pred.shape[-1], target.shape[-1])
    return loss_fn(pred[..., :min_dim], target[..., :min_dim])

for epoch in range(num_epochs):
    total_loss = 0
    for feats, imgs, targets, mask in loader:
        # feats, imgs, targets, mask = feats.to(device), imgs.to(device), targets.to(device), mask.to(device)

        # print(feats.shape)
        pred, _ = model(feats, imgs, mask)
        # break
        # loss = loss_fn(pred, targets)
        loss = safe_loss(pred, targets, loss_fn)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss={total_loss/len(loader):.4f}')

feats, imgs, targets, mask = next(iter(loader))
_, lstm_out = model(feats, imgs, mask)
plt.imshow(lstm_out[0].detach().numpy().T, aspect='auto', cmap='hot')
plt.title('Time Cell Activation (Hidden States)')
plt.xlabel('Time Step')
plt.ylabel('Hidden Unit')
plt.show()


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import seaborn as sns

def analyze_time_cells(model, loader, n_cells_to_plot=16):
    model.eval()
    all_hidden_states = []

    # ---- 收集所有轨迹的 LSTM 输出 ----
    with torch.no_grad():
        for feats, imgs, targets, mask in loader:
            # feats = feats.to(device)
            # imgs = imgs.to(device)
            # mask = mask.to(device)
            _, lstm_out = model(feats, imgs, mask)
            all_hidden_states.extend(list(lstm_out.cpu().numpy()))  # 每个 batch 样本单独存

    # ---- 自动 Padding 到相同时间长度 ----
    max_T = max([x.shape[0] for x in all_hidden_states])
    hidden_dim = all_hidden_states[0].shape[-1]
    padded_states = np.zeros((len(all_hidden_states), max_T, hidden_dim))
    for i, h in enumerate(all_hidden_states):
        T = h.shape[0]
        padded_states[i, :T, :] = h

    hidden_concat = padded_states  # [N_seq, T_max, H]
    print(f"✅ Hidden states padded to {hidden_concat.shape}")

    # ---- 1️⃣ 热图（单条轨迹时间细胞激活） ----
    plt.figure(figsize=(10,6))
    sns.heatmap(hidden_concat[0].T, cmap='inferno', cbar=True)
    plt.title("Example LSTM Hidden State Heatmap (Time Cell Activation)")
    plt.xlabel("Time step")
    plt.ylabel("Hidden unit")
    plt.show()

    # ---- 2️⃣ 峰值时间分布 ----
    hidden_mean = hidden_concat.mean(axis=0)  # [T, H]
    peak_times = hidden_mean.argmax(axis=0)
    plt.figure(figsize=(6,4))
    plt.hist(peak_times, bins=20, color='orange', edgecolor='k')
    plt.title("Distribution of Peak Activation Times")
    plt.xlabel("Time step of max activation")
    plt.ylabel("Cell count")
    plt.show()

    # ---- 3️⃣ 时间选择性指数 (TSI) ----
    cell_mean = hidden_concat.mean(axis=(0,1))
    cell_max = hidden_concat.max(axis=(0,1))
    cell_std = hidden_concat.std(axis=(0,1)) + 1e-6
    tsi = (cell_max - cell_mean) / cell_std
    plt.figure(figsize=(8,3))
    plt.bar(np.arange(len(tsi)), tsi)
    plt.title("Temporal Selectivity Index (TSI) per Cell")
    plt.xlabel("Cell index")
    plt.ylabel("TSI value")
    plt.show()

    # ---- 4️⃣ 多尺度时间跨度统计 ----
    cell_timescale = []
    for i in range(hidden_concat.shape[-1]):
        act_curve = hidden_concat[0,:,i]
        above_thresh = act_curve > (act_curve.max() * 0.5)
        duration = above_thresh.sum()
        cell_timescale.append(duration)
    plt.figure(figsize=(6,4))
    plt.hist(cell_timescale, bins=15, color='teal', edgecolor='k')
    plt.title("Time-Scale Diversity of Time Cells")
    plt.xlabel("Active duration (steps > 0.5·max)")
    plt.ylabel("Cell count")
    plt.show()

    # ---- 5️⃣ PCA 可视化 ----
    flat_hidden = hidden_concat.reshape(-1, hidden_concat.shape[-1])
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(flat_hidden)
    plt.figure(figsize=(5,5))
    plt.scatter(reduced[:,0], reduced[:,1], s=2, alpha=0.4, c=np.linspace(0,1,len(reduced)))
    plt.title("PCA projection of Time Cell Activity")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

    print("多尺度时间细胞评估完成。")
    return {
        "hidden_concat": hidden_concat,
        "peak_times": peak_times,
        "tsi": tsi,
        "cell_timescale": cell_timescale
    }

# ✅ 运行评估
results = analyze_time_cells(model, loader)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import mean_squared_error
from scipy.ndimage import gaussian_filter1d
import warnings, os

# ---------- 安全版 MSE 计算 ----------
def safe_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)
    if y_true.shape[0] != y_pred.shape[0]:
        raise ValueError(f"Sample count mismatch: {y_true.shape[0]} vs {y_pred.shape[0]}")
    D_true, D_pred = y_true.shape[1], y_pred.shape[1]
    D_common = min(D_true, D_pred)
    if D_true != D_pred:
        warnings.warn(f"Dimension mismatch: y_true={D_true}, y_pred={D_pred}. Using first {D_common} dims.")
    mse = np.mean((y_true[:, :D_common] - y_pred[:, :D_common])**2)
    return mse

# ---------- 主评估函数 ----------
def evaluate_time_scales_and_extensions(model, loader, results, thresholds=(10, 30), save_dir="./eval_results"):
    os.makedirs(save_dir, exist_ok=True)

    hidden_concat = results['hidden_concat']
    cell_timescale = np.array(results['cell_timescale'])
    fast_thr, slow_thr = thresholds

    fast_idx = np.where(cell_timescale <= fast_thr)[0]
    medium_idx = np.where((cell_timescale > fast_thr) & (cell_timescale <= slow_thr))[0]
    slow_idx = np.where(cell_timescale > slow_thr)[0]
    print(f"Fast={len(fast_idx)}, Medium={len(medium_idx)}, Slow={len(slow_idx)}")

    # ---------- 安全消融 ----------
    def ablate_cells(model, mask_idx):
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for feats, imgs, targets, mask in loader:
                pred, lstm_out = model(feats, imgs, mask)
                if len(mask_idx) > 0:
                    lstm_out[:,:,mask_idx] = 0
                new_pred = model.fc(lstm_out[range(lstm_out.size(0)), mask.sum(1)-1])
                y_true.extend(targets.cpu().numpy())
                y_pred.extend(new_pred.cpu().numpy())
        return safe_mean_squared_error(y_true, y_pred)

    mse_base = ablate_cells(model, [])
    mse_fast = ablate_cells(model, fast_idx)
    mse_med = ablate_cells(model, medium_idx)
    mse_slow = ablate_cells(model, slow_idx)

    # ---------- 绘图 ----------
    plt.figure(figsize=(6,4))
    categories = ['Baseline', 'No Fast', 'No Medium', 'No Slow']
    mse_values = [mse_base, mse_fast, mse_med, mse_slow]
    plt.bar(categories, mse_values, color=['gray','red','orange','green'])
    plt.ylabel("MSE (↓ better)")
    plt.title("📉 Time-Scale Ablation Analysis")
    for i,v in enumerate(mse_values): plt.text(i, v + 0.0002, f"{v:.4f}", ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "ablation_analysis.png"), dpi=200)
    plt.show()

    # ---------- 时间细胞相关性 ----------
    hidden_mean = hidden_concat.mean(axis=0)
    corr = np.corrcoef(hidden_mean.T)
    plt.figure(figsize=(6,5))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title("🔍 Correlation Matrix between Time Cells")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "correlation_matrix.png"), dpi=200)
    plt.show()

    def avg_corr(a,b): return np.nanmean(np.abs(corr[np.ix_(a,b)]))
    corr_fm, corr_ms, corr_fs = avg_corr(fast_idx,medium_idx), avg_corr(medium_idx,slow_idx), avg_corr(fast_idx,slow_idx)

    # ---------- 多尺度滤波模拟 ----------
    plt.figure(figsize=(10,4))
    base_curve = hidden_concat[0,:,0]
    for s in [1,3,6]:
        plt.plot(gaussian_filter1d(base_curve, sigma=s), label=f"σ={s}")
    plt.title("🔁 Multi-scale Temporal Filtering")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "multi_scale_filtering.png"), dpi=200)
    plt.show()

    # ---------- 跨轨迹相关性 ----------
    if hidden_concat.shape[0] >= 2:
        corr_cross = np.corrcoef(hidden_concat[0].T, hidden_concat[1].T)[:hidden_concat.shape[2], hidden_concat.shape[2]:]
        plt.figure(figsize=(5,5))
        sns.heatmap(corr_cross, cmap='viridis')
        plt.title("📈 Cross-Trajectory Time Cell Correlation")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "cross_traj_corr.png"), dpi=200)
        plt.show()

    # ---------- 保存汇总表 ----------
    df = pd.DataFrame({
        "metric": ["MSE_baseline", "MSE_no_fast", "MSE_no_medium", "MSE_no_slow",
                   "Corr_Fast_Med", "Corr_Med_Slow", "Corr_Fast_Slow"],
        "value": [mse_base, mse_fast, mse_med, mse_slow, corr_fm, corr_ms, corr_fs]
    })
    csv_path = os.path.join(save_dir, "timecell_eval_summary.csv")
    df.to_csv(csv_path, index=False)
    print(f"✅ Saved evaluation summary to {csv_path}")

    return {"mse_base": mse_base, "mse_fast": mse_fast, "mse_med": mse_med, "mse_slow": mse_slow,
            "corrs": (corr_fm, corr_ms, corr_fs), "csv": csv_path}

# ✅ 运行
extended_results = evaluate_time_scales_and_extensions(model, loader, results)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from scipy.stats import pearsonr

def evaluate_timecell_trajectory_correlation(model, loader, results, save_dir="./eval_results"):
    os.makedirs(save_dir, exist_ok=True)
    hidden_concat = results['hidden_concat']   # [N_seq, T, H]

    # ---- Step 1: 收集轨迹数据（自动 Padding）----
    all_positions, all_yaws = [], []
    for feats, imgs, targets, mask in loader:
        feats_np = feats.cpu().numpy()  # [B, T, 4]
        pos = feats_np[..., :3]
        yaw = feats_np[..., 3] if feats_np.shape[-1] > 3 else np.zeros_like(pos[..., 0])
        all_positions.extend(list(pos))
        all_yaws.extend(list(yaw))

    # 计算最大长度并补齐
    T_max = max(p.shape[0] for p in all_positions)
    padded_pos = np.zeros((len(all_positions), T_max, 3))
    padded_yaw = np.zeros((len(all_yaws), T_max))
    for i, (p, y) in enumerate(zip(all_positions, all_yaws)):
        T = p.shape[0]
        padded_pos[i, :T] = p
        padded_yaw[i, :T] = y

    pos_all, yaw_all = padded_pos, padded_yaw
    print(f"✅ Padded trajectory data to shape: pos={pos_all.shape}, yaw={yaw_all.shape}")

    # ---- Step 2: 计算时间细胞-位置相关性 ----
    n_cells = hidden_concat.shape[-1]
    corr_pos = np.zeros((n_cells, 3))
    for c in range(n_cells):
        h = hidden_concat[..., c].flatten()
        for dim in range(3):
            corr_pos[c, dim] = pearsonr(h, pos_all[..., dim].flatten())[0]
    corr_pos_mean = np.mean(np.abs(corr_pos), axis=1)

    # ---- Step 3: 计算时间细胞-yaw相关性 ----
    corr_yaw = np.zeros(n_cells)
    for c in range(n_cells):
        h = hidden_concat[..., c].flatten()
        corr_yaw[c] = pearsonr(h, yaw_all.flatten())[0]

    # ---- Step 4: 可视化 ----
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.bar(np.arange(n_cells), np.abs(corr_pos_mean), color='skyblue')
    plt.title("📍 Position correlation per cell")
    plt.xlabel("Time cell index")
    plt.ylabel("|r| (pos)")
    plt.subplot(1,2,2)
    plt.bar(np.arange(n_cells), np.abs(corr_yaw), color='orange')
    plt.title("🧭 Yaw correlation per cell")
    plt.xlabel("Time cell index")
    plt.ylabel("|r| (yaw)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "cell_position_yaw_corr.png"), dpi=200)
    plt.show()

    # ---- Step 5: Top correlated cells ----
    top_k = 10
    top_pos_idx = np.argsort(-np.abs(corr_pos_mean))[:top_k]
    top_yaw_idx = np.argsort(-np.abs(corr_yaw))[:top_k]
    print(f"Top {top_k} position-correlated cells:", top_pos_idx)
    print(f"Top {top_k} yaw-correlated cells:", top_yaw_idx)

    # ---- Step 6: 速度相关性 ----
    velocity = np.linalg.norm(np.diff(pos_all, axis=1, prepend=pos_all[:, :1]), axis=-1)
    corr_vel = np.zeros(n_cells)
    for c in range(n_cells):
        h = hidden_concat[..., c].flatten()
        corr_vel[c] = pearsonr(h, velocity.flatten())[0]

    plt.figure(figsize=(8,4))
    plt.bar(np.arange(n_cells), np.abs(corr_vel), color='teal')
    plt.title("🏃 Time cell - velocity correlation")
    plt.xlabel("Cell index")
    plt.ylabel("|r| (velocity)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "cell_velocity_corr.png"), dpi=200)
    plt.show()

    # ---- Step 7: 保存 CSV ----
    df_corr = pd.DataFrame({
        "cell_index": np.arange(n_cells),
        "corr_x": corr_pos[:,0],
        "corr_y": corr_pos[:,1],
        "corr_z": corr_pos[:,2],
        "corr_pos_mean": corr_pos_mean,
        "corr_yaw": corr_yaw,
        "corr_velocity": corr_vel
    })
    csv_path = os.path.join(save_dir, "timecell_traj_corr.csv")
    df_corr.to_csv(csv_path, index=False)
    print(f"✅ Saved cell-trajectory correlation results to {csv_path}")

    # ---- Step 8: 热图 ----
    plt.figure(figsize=(8,6))
    sns.heatmap(np.stack([corr_pos_mean, corr_yaw, corr_vel], axis=1).T,
                cmap="coolwarm", center=0,
                yticklabels=["Pos", "Yaw", "Velocity"])
    plt.title("🔥 Time cell - Trajectory Correlation Heatmap")
    plt.xlabel("Cell index")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "cell_traj_corr_heatmap.png"), dpi=200)
    plt.show()

    return df_corr

# ✅ 调用轨迹相关评估
df_corr = evaluate_timecell_trajectory_correlation(model, loader, results)
